### Lab Assignment: Commercial Data Analysis

### University of Virginia
### DS 5110: Big Data Systems
### Last Updated: February 15, 2026

---

### INSTRUCTIONS  
In this assignment, you will work with a dataset containing information about businesses.  
Each record is a business location.  Follow the steps below, writing and running the code in blocks, and displaying the solutions.  

The path to the dataset is in a file named `find_dataset_on_rivanna.txt` in Module 3. 

Each question part is worth 1 POINT, for a total of 15 POINTS.

Hint: reaching deeper fields in json hierarchy can be done like this:  

`df.select('address.street_number')`

---

In [10]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
        .appName("comm") \
        .getOrCreate()

In [1]:
# note that read.json can read a zipped JSON directly

**1. (1 PT) Read in the dataset and show the number of records**

In [12]:
with open("find_dataset_on_rivanna.txt") as f:
    dataset_path = f.readlines()[-1].strip()

df = spark.read.json(dataset_path)

In [13]:
df.count()

154679

**2. (1 PT) Show the first 3 records**

In [49]:
df.show(3, truncate=True)

+--------------------+--------------------+--------------------+----------------+----+--------------------+--------------------+--------------------+
|             address|       business_tags|               hours|              id|menu|             reviews|                urls|             webpage|
+--------------------+--------------------+--------------------+----------------+----+--------------------+--------------------+--------------------+
|{Woodburn, {45.15...|                NULL|                NULL|000023995a540868|NULL|                  []|{woodburn.k12.or....|{Educational Tech...|
|{Hialeah, {25.884...|{[], [{has_atm, Y...|{NULL, 1900, NULL...|0000821a1394916e|NULL|                NULL|{NULL, [yelp.com]...|                NULL|
|{Rochester, {43.1...|{[], [{accepts_cr...|{NULL, 1700, NULL...|000136e65d50c3b7|NULL|[{New (to me) qui...|{usps.com, [yelp....|{Welcome | USPS G...|
+--------------------+--------------------+--------------------+----------------+----+--------------

**3. (1 PT) Show the first 5 street addresses which are not null**  

In [57]:
df.filter(df.address.street.isNotNull() & df.address.street_number.isNotNull()) \
  .select("address.street_number", "address.street") \
  .show(5, truncate=False)

+-------------+------------+
|street_number|street      |
+-------------+------------+
|965          |Boones Ferry|
|1137         |68th        |
|1614         |Penfield    |
|846          |Park        |
|403          |Main        |
+-------------+------------+
only showing top 5 rows


**4. (1 PT) Location**  

Count the number of records where the city is New York

In [52]:
df.filter(df.address.city == "New York").count()

1953

**5. (1 PT) Hours**  

Count the number of records where closing time on Tuesday is 8pm

In [53]:
df.filter(df.hours.tuesday_close == "2000").count()

3154

**6. (1 PT) Location and Hours**  

For the records where the city is New York, aggregate by Tuesday closing time, showing a column called `count` with the number of records for each Tuesday closing time. Sort by the `count` column in descending order. Here is an example of results output:

+-------------+-----+  
|tuesday_close|count|  
+-------------+-----+  
|         1800| 2001|  
|         1700| 1800|  
|         2000| 1550|  

In [25]:
from pyspark.sql.functions import count

df.filter(df.address.city == "New York") \
  .groupBy("hours.tuesday_close") \
  .agg(count("*").alias("count")) \
  .orderBy("count", ascending=False) \
  .show()

[Stage 17:>                                                         (0 + 1) / 1]

+-------------+-----+
|tuesday_close|count|
+-------------+-----+
|         NULL| 1383|
|         1700|  174|
|         1800|   92|
|         1900|   57|
|         2000|   48|
|         2100|   35|
|         2300|   19|
|         2200|   18|
|         0000|   15|
|         1830|   14|
|         1730|   12|
|         1600|   10|
|         1930|    8|
|         2030|    7|
|         0100|    5|
|         2330|    5|
|         2359|    5|
|         0400|    5|
|         2230|    5|
|         1500|    4|
+-------------+-----+
only showing top 20 rows


**7. (1 PT) Price Range**  

Price range is quoted in number of dollar signs.  Count the number of records with price range greater than or equal to three.

In [26]:
from pyspark.sql.functions import col

df.filter(col("menu.price_range").cast("int") >= 3).count()

115

**8. (1 PT) Missing Webpage URL**  

Count the number of records that are missing the webpage url.

In [27]:
df.filter(df.webpage.url.isNull()).count()

79813

**9. (1 PT) Webpage URLs**  

Register the dataframe as a temp table.  
Next, use Spark SQL to select only the webpage title column, filtering on rows where the webpage url (accessed under `webpage.url`) is *Target.com*. 

Show only one resulting row and don't truncate the output.

In [28]:
df.createOrReplaceTempView("business")

In [54]:
spark.sql("""
SELECT webpage.title
FROM business
WHERE webpage.url = 'Target.com'
LIMIT 1
""").show(truncate=False)

+-------------------------------+
|title                          |
+-------------------------------+
|Target : Expect More. Pay Less.|
+-------------------------------+



**10. (1 PT) Analysis on Ratings**  

The reviews contains information such as the number of stars for each review (the *rating*).  
The ratings are stored in an array (`reviews.stars`) for each business location (you should check for yourself). Return the top five most common rating arrays.  For example, an array might look like: 
[5, 5]



In [30]:
from pyspark.sql.functions import expr

df2 = df.withColumn(
    "stars_array",
    expr("transform(reviews, x -> x.stars)")
)

df2.groupBy("stars_array") \
   .count() \
   .orderBy("count", ascending=False) \
   .show(5, truncate=False)

[Stage 27:>                                                         (0 + 1) / 1]

+-----------+-----+
|stars_array|count|
+-----------+-----+
|NULL       |74679|
|[]         |42419|
|[5]        |4258 |
|[NULL]     |3067 |
|[5, 5]     |1610 |
+-----------+-----+
only showing top 5 rows


**11. More work with Ratings**  

For this question, you will filter out null ratings and then compute the average rating for each business location (using the field: `id`).


a) (1 PT) Create a new dataframe retaining two fields: `id`, `reviews.stars`


In [35]:
df_ratings = df.select("id", "reviews.stars")

b) (1 PT) Create a row for each rating  
hint: use the `withColumn()` and `explode()` functions  
you will need to import the `explode()` function by issuing:

`from pyspark.sql.functions import explode`


In [56]:
from pyspark.sql.functions import explode

df_exploded = df_ratings.withColumn("stars", explode("stars"))

c) (1 PT) Return a count of the number of ratings in this dataframe

In [46]:
df_exploded.count()

600082

d) (1 PT) Drop rows where the rating is null, and return a count of the number of non-null ratings

In [47]:
df_clean_ratings = df_exploded.filter(col("stars").isNotNull())

In [48]:
print(df_clean_ratings.count())

[Stage 49:>                                                         (0 + 1) / 1]

538241


e) (1 PT) Compute the average rating, grouped by `id`. After the average is computed, sort by `id` in ascending order and show the top 10 records.  
 
hint:   
this can all be done in one line using the `agg()` function  
this `id` should be at the top: 000136e65d50c3b7

In [40]:
from pyspark.sql.functions import avg

df_clean_ratings.groupBy("id") \
                .agg(avg("stars").alias("avg_rating")) \
                .sort(col("id").asc()) \
                .show(10, truncate=False)

[Stage 37:>                                                         (0 + 1) / 1]

+----------------+------------------+
|id              |avg_rating        |
+----------------+------------------+
|000136e65d50c3b7|4.0               |
|0003b7589a4e12a0|5.0               |
|00059519f0dba1b4|3.3333333333333335|
|000a1df4c8e0ecd2|4.6               |
|000c7b7a30623083|5.0               |
|000c9ffc8b89af03|3.0               |
|000de20baa847ecc|1.6666666666666667|
|001064359d9f162f|5.0               |
|0010c9f495d87dd7|3.0               |
|0017774db5e6400a|4.333333333333333 |
+----------------+------------------+
only showing top 10 rows
